# __ReAct (Resoning and Act) Agent__

### Creating a ReAct Agent

In [ ]:
from langchain_classic.hub import pull
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_experimental.tools import PythonREPLTool
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.5,
)

template = '''

Answer the following questions as best you can.
questions: {q}

'''

prompt_template = PromptTemplate.from_template(template)

prompt = pull('hwchase17/react')



# 1 python REPL tool (for executing python code)

python_repl = PythonREPLTool()

python_repl_tool = Tool(
    name="python_repl",
    func=python_repl.run,
    description="A Python shell. Use this to execute python code. If you want to use any python library, you can import it here. Always use this tool for any python related questions.",
)

# 1 wikipedia tool (for getting information from wikipedia)

api_wrapper = WikipediaAPIWrapper()

wikepedia = WikipediaQueryRun(api_wrapper=api_wrapper)

wikepedia_tool = Tool(
    name="wikipedia",
    func=wikepedia.run,
    description="A wikipedia search tool. Use this to get information from wikipedia.",
)

# 1 duckduckgo search tool (for getting information from the web)

duckduckgo = DuckDuckGoSearchRun()
duckduckgo_tool = Tool(
    name="duckduckgo",
    func=duckduckgo.run,
    description="A search engine. Use this to get information from the web.",
)

tools = [python_repl_tool, wikepedia_tool, duckduckgo_tool]

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, max_iterations=10)

In [13]:
question = 'Generate a python function to calculate the factorial of a number?'
# question = 'What is the capital of India?'
# question = 'who is president of India?'
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})
print(output)



> Entering new AgentExecutor chain...
To generate a Python function that calculates the factorial of a number, we need to use the python_repl tool. 

Action: python_repl
Action Input: 
```
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

print(factorial(5))
```120
Thought: The python_repl tool has successfully executed the function and returned the result. The function takes an integer as input and calculates its factorial using recursion. 

However, to fully address the question, it would be beneficial to provide the function itself as the answer, along with an explanation of how it works.

Action: None needed, as we already have the function.

Thought: I now know the final answer.

Final Answer: The Python function to calculate the factorial of a number is:
```
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```
This function uses recursion to calculate the factorial. For example, to calcula